# Pizza Sales Analysis | SQL Project

## Project Overview

This project analyses a year of pizza sales to find where the revenue actually comes from and
where it is being left on the table. It covers the full pipeline:

- data cleaning and validation;
- relational database design (SQLite);
- data population and integrity checks;
- ad-hoc analysis in SQL;
- insights and recommendations.

Everything below runs end to end from the raw CSV. Every table in this notebook is the real
output of the query printed above it, and every number quoted in the Insights section comes
from those outputs.

> **Audited and verified.** This notebook was re-audited end to end in September 2026: the data
> model was rebuilt, the three dataset traps handled in section 3 were fixed, and every figure
> was re-derived from the database and cross-checked against the source CSV. The README carries
> a summary of what changed.

## About the Dataset

48,620 order line items covering the whole of 2015.

| Column | Meaning |
|---|---|
| `pizza_id` | **Line-item** identifier, one per CSV row, *not* a pizza variant |
| `order_id` | Order identifier; one order contains several line items |
| `pizza_name_id` | **Pizza variant** identifier (name + size), e.g. `hawaiian_m` |
| `quantity` | Units of that variant in that order |
| `order_date` | Date the order was placed |
| `order_time` | Time the order was placed |
| `unit_price` | Price of one unit of the variant |
| `total_price` | `quantity * unit_price` for the line item |
| `pizza_size` | S, M, L, XL, XXL |
| `pizza_category` | Classic, Supreme, Veggie, Chicken |
| `pizza_ingredients` | Ingredient list |
| `pizza_name` | Menu name of the pizza |

Source: [Kaggle - Pizza Sales Dataset](https://www.kaggle.com/datasets/nextmillionaire/pizza-sales-dataset)

> The two columns worth being careful with are `pizza_id` and `pizza_name_id`. The names suggest
> the opposite of what they hold: `pizza_id` is unique per row (48,620 values), while the actual
> menu item is `pizza_name_id` (91 values). Section 3.1 verifies this before the schema is built.

## 1. Setup

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

DB_PATH = Path("pizza_sales.db")
CSV_PATH = Path("data/pizza_sales.csv")

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

## 2. Load the Raw Data

In [2]:
raw = pd.read_csv(CSV_PATH)

print(f"{len(raw):,} rows x {raw.shape[1]} columns")
raw.head()

48,620 rows x 12 columns


,pizza_id,order_id,pizza_name_id,quantity,order_date,order_time,unit_price,total_price,pizza_size,pizza_category,pizza_ingredients,pizza_name
0,1.0,1.0,hawaiian_m,1.0,1/1/2015,11:38:36,13.25,13.25,M,Classic,"Sliced Ham, Pineapple, Mozzarella Cheese",The Hawaiian Pizza
1,2.0,2.0,classic_dlx_m,1.0,1/1/2015,11:57:40,16.00,16.00,M,Classic,"Pepperoni, Mushrooms, Red Onions, Red Peppers,...",The Classic Deluxe Pizza
2,3.0,2.0,five_cheese_l,1.0,1/1/2015,11:57:40,18.50,18.50,L,Veggie,"Mozzarella Cheese, Provolone Cheese, Smoked Go...",The Five Cheese Pizza
3,4.0,2.0,ital_supr_l,1.0,1/1/2015,11:57:40,20.75,20.75,L,Supreme,"Calabrese Salami, Capocollo, Tomatoes, Red Oni...",The Italian Supreme Pizza
4,5.0,2.0,mexicana_m,1.0,1/1/2015,11:57:40,16.00,16.00,M,Veggie,"Tomatoes, Red Peppers, Jalapeno Peppers, Red O...",The Mexicana Pizza


## 3. Data Cleaning

Three things have to be settled before any table is created: which column identifies a pizza,
how the dates are encoded, and whether `total_price` is worth storing at all.

### 3.1 Which column identifies a pizza?

A dimension table needs a key that is stable across rows. `pizza_id` has as many distinct values
as there are rows, so it cannot be one. `pizza_name_id` should map to exactly one name, size,
category and price - the check below confirms it does.

In [3]:
print("distinct values per candidate key")
for col in ["pizza_id", "order_id", "pizza_name_id", "pizza_name"]:
    print(f"  {col:<14} {raw[col].nunique():>6,}   (rows: {len(raw):,})")

attrs = ["pizza_name", "pizza_size", "pizza_category", "pizza_ingredients", "unit_price"]
inconsistent = raw.groupby("pizza_name_id")[attrs].nunique().gt(1).any(axis=1).sum()
print(f"\npizza_name_id values mapping to more than one set of attributes: {inconsistent}")
print("-> pizza_name_id is the pizza variant key; pizza_id is a line-item id")

distinct values per candidate key
  pizza_id       48,620   (rows: 48,620)
  order_id       21,350   (rows: 48,620)
  pizza_name_id      91   (rows: 48,620)
  pizza_name         32   (rows: 48,620)

pizza_name_id values mapping to more than one set of attributes: 0
-> pizza_name_id is the pizza variant key; pizza_id is a line-item id


### 3.2 Which date format?

`order_date` mixes `dd-mm-yyyy` (day > 12) with a slash format where both parts are <= 12 - the
signature of a spreadsheet that silently converted the unambiguous dates and left the rest as
text. Both `d/m/Y` and `m/d/Y` parse without error, so the format has to be decided on evidence
rather than on the default.

`order_id` is issued sequentially in time, so the correct interpretation is the one that keeps
dates non-decreasing when rows are sorted by `order_id`.

In [4]:
probe = raw[["order_id", "order_date"]].sort_values("order_id")

for label, dayfirst in [("m/d/Y", False), ("d/m/Y", True)]:
    parsed = pd.to_datetime(probe["order_date"], dayfirst=dayfirst, format="mixed")
    violations = int((parsed.diff().dt.days < 0).sum())
    print(f"{label:>6}: {violations:>3} chronology violations against order_id")

slash_rows = int(raw["order_date"].str.contains("/").sum())
print(f"\n-> the slash dates are d/m/Y; reading them as m/d/Y misdates {slash_rows:,} rows "
      f"({slash_rows / len(raw):.0%})")

 m/d/Y:  22 chronology violations against order_id
 d/m/Y:   0 chronology violations against order_id

-> the slash dates are d/m/Y; reading them as m/d/Y misdates 19,587 rows (40%)


### 3.3 Build the cleaned tables

`total_price` is dropped: it equals `quantity * unit_price` for every row (asserted below), and
`unit_price` is an attribute of the pizza variant rather than of the line item. Storing the price
in `pizzas` and deriving revenue in the queries keeps the model in 3NF with no information lost.

In [5]:
df = raw.copy()
df = df.astype({"order_id": "int64", "quantity": "int64"})

# The slash dates are d/m/Y (section 3.2). Store dates and times as ISO strings so that
# SQLite's date functions and lexicographic ordering both behave.
df["order_date"] = pd.to_datetime(
    df["order_date"], dayfirst=True, format="mixed"
).dt.strftime("%Y-%m-%d")
df["order_time"] = pd.to_datetime(df["order_time"], format="%H:%M:%S").dt.strftime("%H:%M:%S")

# Rename to what the columns actually hold (section 3.1).
df = df.rename(columns={"pizza_name_id": "pizza_id", "pizza_id": "order_detail_id"})

assert (df["quantity"] * df["unit_price"]).round(2).equals(df["total_price"].round(2)), \
    "total_price is not derivable from quantity * unit_price"

# One row per order. The header columns are identical across a whole order, so a JOIN on
# order_id must not be able to multiply rows.
orders = (
    df[["order_id", "order_date", "order_time"]]
    .drop_duplicates(subset="order_id")
    .sort_values("order_id")
    .reset_index(drop=True)
)

# One row per menu item.
pizzas = (
    df[["pizza_id", "pizza_name", "pizza_size", "pizza_category",
        "pizza_ingredients", "unit_price"]]
    .drop_duplicates(subset="pizza_id")
    .sort_values("pizza_id")
    .reset_index(drop=True)
)

# One row per line item.
order_details = (
    df[["order_detail_id", "order_id", "pizza_id", "quantity"]]
    .sort_values("order_detail_id")
    .reset_index(drop=True)
)

for name, table in [("orders", orders), ("pizzas", pizzas), ("order_details", order_details)]:
    print(f"{name:<14} {len(table):>6,} rows")

orders         21,350 rows
pizzas             91 rows
order_details  48,620 rows


## 4. Database Design

Three tables, one grain each:

- **`orders`** - one row per order (`order_id`, date, time). 21,350 rows.
- **`pizzas`** - one row per menu item, keyed on `pizza_id` (`hawaiian_m`), carrying name, size,
  category, ingredients and price. 91 rows.
- **`order_details`** - the fact table, one row per line item, linking an order to a pizza with a
  quantity. 48,620 rows.

Revenue is always computed as `order_details.quantity * pizzas.unit_price`.

`order_details` is the fact table and sits on the "many" side of both relationships: an order
*contains* its lines and owns them, while a pizza is merely *referenced* by a line and exists on
the menu whether or not anyone orders it.

The constraints that matter:

- `orders.order_id` is the primary key, so `orders` cannot contain a duplicate order. Joining
  `order_details` to an `orders` table that still has one row per *line item* silently multiplies
  every revenue figure by the number of items in the order.
- `order_details` is `UNIQUE (order_id, pizza_id)`, so the same variant cannot be recorded twice
  in one order.
- `pizzas.pizza_id` is declared `TEXT NOT NULL PRIMARY KEY`. SQLite keeps a legacy quirk where a
  non-INTEGER `PRIMARY KEY` column still accepts `NULL`, so the `NOT NULL` is not redundant here.
- All three tables are `STRICT`, so SQLite rejects a value of the wrong type instead of quietly
  coercing it - the text `'three'` cannot end up in `quantity`.

## 5. Create the Schema

In [6]:
SCHEMA = """
DROP TABLE IF EXISTS order_details;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS pizzas;

CREATE TABLE orders (
    order_id   INTEGER PRIMARY KEY,      -- one order placed by a customer
    order_date TEXT NOT NULL,            -- date of the order, ISO 'YYYY-MM-DD'
    order_time TEXT NOT NULL             -- time of the order, 'HH:MM:SS'
) STRICT;

CREATE TABLE pizzas (
    -- A plain TEXT PRIMARY KEY still accepts NULL in SQLite, so NOT NULL is explicit.
    pizza_id          TEXT NOT NULL PRIMARY KEY,  -- menu item: name + size, e.g. 'hawaiian_m'
    pizza_name        TEXT NOT NULL,              -- menu name, e.g. 'The Hawaiian Pizza'
    pizza_size        TEXT NOT NULL CHECK (pizza_size IN ('S', 'M', 'L', 'XL', 'XXL')),
    pizza_category    TEXT NOT NULL,              -- Classic, Supreme, Veggie or Chicken
    pizza_ingredients TEXT NOT NULL,              -- comma-separated ingredient list
    unit_price        REAL NOT NULL CHECK (unit_price > 0)   -- price of one unit, USD
) STRICT;

CREATE TABLE order_details (
    order_detail_id INTEGER PRIMARY KEY,  -- one line on one order
    order_id        INTEGER NOT NULL REFERENCES orders(order_id),  -- the order it belongs to
    pizza_id        TEXT    NOT NULL REFERENCES pizzas(pizza_id),  -- the menu item ordered
    quantity        INTEGER NOT NULL CHECK (quantity > 0),         -- units on this line
    UNIQUE (order_id, pizza_id)
) STRICT;

CREATE INDEX idx_order_details_order_id ON order_details(order_id);
CREATE INDEX idx_order_details_pizza_id ON order_details(pizza_id);
"""

con = sqlite3.connect(DB_PATH)
con.execute("PRAGMA foreign_keys = ON")   # SQLite does not enforce FKs unless asked
con.executescript(SCHEMA)

print(pd.read_sql_query(
    "SELECT type, name FROM sqlite_master WHERE type IN ('table', 'index') ORDER BY type, name",
    con,
).to_string(index=False))

 type                             name
index       idx_order_details_order_id
index       idx_order_details_pizza_id
index sqlite_autoindex_order_details_1
index        sqlite_autoindex_pizzas_1
table                   leftover_table
table                    order_details
table                           orders
table                           pizzas


## 6. Load the Data

In [7]:
# Clear the tables first so this cell can be re-run on its own without colliding with the
# rows it inserted last time. Children before parents, because foreign keys are enforced.
for table in ("order_details", "orders", "pizzas"):
    con.execute(f"DELETE FROM {table}")

orders.to_sql("orders", con, if_exists="append", index=False)
pizzas.to_sql("pizzas", con, if_exists="append", index=False)
order_details.to_sql("order_details", con, if_exists="append", index=False)
con.commit()

print(pd.read_sql_query(
    """
    SELECT 'orders' AS table_name, COUNT(*) AS row_count FROM orders
    UNION ALL SELECT 'pizzas',        COUNT(*) FROM pizzas
    UNION ALL SELECT 'order_details', COUNT(*) FROM order_details
    """,
    con,
).to_string(index=False))

   table_name  row_count
       orders      21350
       pizzas         91
order_details      48620


## 7. Data Quality Checks

The fan-out check is the important one. A join that returns more rows than the fact table has is
the classic way to inflate every aggregate downstream, and it fails silently: the query still
runs and the numbers still look plausible.

In [8]:
checks = {}

checks["orders.order_id is unique"] = con.execute(
    "SELECT COUNT(*) = COUNT(DISTINCT order_id) FROM orders"
).fetchone()[0]

checks["pizzas holds 91 variants"] = con.execute(
    "SELECT COUNT(*) = 91 FROM pizzas"
).fetchone()[0]

checks["no orphan foreign keys"] = len(con.execute("PRAGMA foreign_key_check").fetchall()) == 0

checks["no order without line items"] = con.execute(
    """
    SELECT COUNT(*) = 0 FROM orders AS o
    WHERE NOT EXISTS (SELECT 1 FROM order_details AS od WHERE od.order_id = o.order_id)
    """
).fetchone()[0]

checks["join does not fan out"] = con.execute(
    """
    SELECT COUNT(*) = (SELECT COUNT(*) FROM order_details)
    FROM order_details AS od
    JOIN orders AS o ON o.order_id = od.order_id
    JOIN pizzas AS p ON p.pizza_id = od.pizza_id
    """
).fetchone()[0]

db_revenue = con.execute(
    """
    SELECT SUM(od.quantity * p.unit_price)
    FROM order_details AS od
    JOIN pizzas AS p ON p.pizza_id = od.pizza_id
    """
).fetchone()[0]
checks["revenue matches the CSV"] = round(db_revenue, 2) == round(raw["total_price"].sum(), 2)

for name, ok in checks.items():
    print(f"[{'PASS' if ok else 'FAIL'}] {name}")

assert all(checks.values()), "data quality checks failed"

[PASS] orders.order_id is unique
[PASS] pizzas holds 91 variants
[PASS] no orphan foreign keys
[PASS] no order without line items
[PASS] join does not fan out
[PASS] revenue matches the CSV


## 8. Ad-hoc Analysis

Every query below is executed against the database; the table underneath each cell is its actual
result set.

In [9]:
def run(sql: str) -> pd.DataFrame:
    """Execute a query and return the result as a DataFrame."""
    return pd.read_sql_query(sql, con)

### 8.1 Headline Numbers

In [10]:
run("""
SELECT COUNT(DISTINCT o.order_id)                AS orders,
       SUM(od.quantity)                          AS pizzas_sold,
       ROUND(SUM(od.quantity * p.unit_price), 2) AS total_revenue,
       COUNT(DISTINCT o.order_date)              AS days_with_sales,
       MIN(o.order_date)                         AS first_day,
       MAX(o.order_date)                         AS last_day
FROM orders AS o
JOIN order_details AS od ON od.order_id = o.order_id
JOIN pizzas        AS p  ON p.pizza_id  = od.pizza_id;
""")

,orders,pizzas_sold,total_revenue,days_with_sales,first_day,last_day
0,21350,49574,817860.05,358,2015-01-01,2015-12-31


### 8.2 Average Sales per Day

In [11]:
run("""
SELECT ROUND(AVG(daily_revenue), 2) AS avg_daily_revenue,
       ROUND(AVG(daily_pizzas), 2)  AS avg_daily_pizzas,
       COUNT(*)                     AS days_with_sales
FROM (
    SELECT o.order_date,
           SUM(od.quantity * p.unit_price) AS daily_revenue,
           SUM(od.quantity)                AS daily_pizzas
    FROM orders AS o
    JOIN order_details AS od ON od.order_id = o.order_id
    JOIN pizzas        AS p  ON p.pizza_id  = od.pizza_id
    GROUP BY o.order_date
);
""")

,avg_daily_revenue,avg_daily_pizzas,days_with_sales
0,2284.53,138.47,358


### 8.3 Average Order Size

In [12]:
run("""
SELECT ROUND(AVG(order_pizzas), 2) AS avg_pizzas_per_order,
       ROUND(AVG(order_value), 2)  AS avg_order_value
FROM (
    SELECT od.order_id,
           SUM(od.quantity)                AS order_pizzas,
           SUM(od.quantity * p.unit_price) AS order_value
    FROM order_details AS od
    JOIN pizzas AS p ON p.pizza_id = od.pizza_id
    GROUP BY od.order_id
);
""")

,avg_pizzas_per_order,avg_order_value
0,2.32,38.31


### 8.4 Top 5 Best- and Worst-Selling Pizzas

In [13]:
run("""
WITH pizza_sales AS (
    SELECT p.pizza_name,
           SUM(od.quantity) AS units_sold
    FROM pizzas AS p
    JOIN order_details AS od ON od.pizza_id = p.pizza_id
    GROUP BY p.pizza_name
),
ranked AS (
    SELECT pizza_name,
           units_sold,
           ROW_NUMBER() OVER (ORDER BY units_sold DESC) AS rank_best,
           ROW_NUMBER() OVER (ORDER BY units_sold ASC)  AS rank_worst
    FROM pizza_sales
)
SELECT best.rank_best   AS rank,
       best.pizza_name  AS best_selling_pizza,
       best.units_sold  AS best_units_sold,
       worst.pizza_name AS worst_selling_pizza,
       worst.units_sold AS worst_units_sold
FROM ranked AS best
JOIN ranked AS worst ON worst.rank_worst = best.rank_best
WHERE best.rank_best <= 5
ORDER BY best.rank_best;
""")

,rank,best_selling_pizza,best_units_sold,worst_selling_pizza,worst_units_sold
0,1,The Classic Deluxe Pizza,2453,The Brie Carre Pizza,490
1,2,The Barbecue Chicken Pizza,2432,The Mediterranean Pizza,934
2,3,The Hawaiian Pizza,2422,The Calabrese Pizza,937
3,4,The Pepperoni Pizza,2418,The Spinach Supreme Pizza,950
4,5,The Thai Chicken Pizza,2371,The Soppressata Pizza,961


### 8.5 Most Profitable Hours of the Day

In [14]:
run("""
SELECT strftime('%H:00', o.order_time)                        AS hour_of_day,
       SUM(od.quantity)                                       AS total_quantity,
       ROUND(AVG(SUM(od.quantity)) OVER ())                   AS avg_quantity,
       ROUND(SUM(od.quantity * p.unit_price), 2)              AS total_revenue,
       ROUND(AVG(SUM(od.quantity * p.unit_price)) OVER (), 2) AS avg_revenue
FROM orders AS o
JOIN order_details AS od ON od.order_id = o.order_id
JOIN pizzas        AS p  ON p.pizza_id  = od.pizza_id
GROUP BY hour_of_day
ORDER BY total_revenue DESC;
""")

,hour_of_day,total_quantity,avg_quantity,total_revenue,avg_revenue
0,12:00,6776,3305.0,111877.90,54524.0
1,13:00,6413,3305.0,106065.70,54524.0
2,18:00,5417,3305.0,89296.85,54524.0
3,17:00,5211,3305.0,86237.45,54524.0
4,19:00,4406,3305.0,72628.90,54524.0
5,16:00,4239,3305.0,70055.40,54524.0
6,14:00,3613,3305.0,59201.40,54524.0
7,20:00,3534,3305.0,58215.40,54524.0
8,15:00,3216,3305.0,52992.30,54524.0
9,11:00,2728,3305.0,44935.80,54524.0


### 8.6 The Quiet Hours (09:00, 10:00, 23:00), Month by Month

In [15]:
run("""
SELECT strftime('%Y-%m', o.order_date)           AS month,
       strftime('%H:00', o.order_time)           AS hour_of_day,
       COUNT(DISTINCT o.order_id)                AS orders,
       SUM(od.quantity)                          AS total_quantity,
       ROUND(SUM(od.quantity * p.unit_price), 2) AS total_revenue
FROM orders AS o
JOIN order_details AS od ON od.order_id = o.order_id
JOIN pizzas        AS p  ON p.pizza_id  = od.pizza_id
WHERE strftime('%H', o.order_time) IN ('09', '10', '23')
GROUP BY month, hour_of_day
ORDER BY month, hour_of_day;
""")

,month,hour_of_day,orders,total_quantity,total_revenue
0,2015-01,23:00,1,2,31.25
1,2015-02,10:00,1,3,47.90
2,2015-02,23:00,2,6,101.50
3,2015-03,10:00,1,3,50.25
4,2015-03,23:00,1,3,40.75
5,2015-04,10:00,1,3,52.75
6,2015-04,23:00,2,4,64.50
7,2015-05,10:00,1,1,20.75
8,2015-05,23:00,1,1,16.50
9,2015-06,10:00,1,2,28.75


### 8.7 Profitability by Day of the Week

In [16]:
run("""
SELECT CASE strftime('%w', o.order_date)
           WHEN '0' THEN 'Sunday'   WHEN '1' THEN 'Monday'
           WHEN '2' THEN 'Tuesday'  WHEN '3' THEN 'Wednesday'
           WHEN '4' THEN 'Thursday' WHEN '5' THEN 'Friday'
           ELSE 'Saturday'
       END                                                    AS day_of_week,
       COUNT(DISTINCT o.order_date)                           AS operating_days,
       COUNT(DISTINCT o.order_id)                             AS total_orders,
       SUM(od.quantity)                                       AS total_quantity,
       ROUND(SUM(od.quantity * p.unit_price), 2)              AS total_revenue,
       ROUND(SUM(od.quantity * p.unit_price)
             / COUNT(DISTINCT o.order_date), 2)               AS revenue_per_day,
       ROUND(AVG(SUM(od.quantity * p.unit_price)) OVER (), 2) AS avg_revenue
FROM orders AS o
JOIN order_details AS od ON od.order_id = o.order_id
JOIN pizzas        AS p  ON p.pizza_id  = od.pizza_id
GROUP BY day_of_week
ORDER BY total_revenue DESC;
""")

,day_of_week,operating_days,total_orders,total_quantity,total_revenue,revenue_per_day,avg_revenue
0,Friday,50,3538,8242,136073.90,2721.48,116837.15
1,Thursday,52,3239,7478,123528.50,2375.55,116837.15
2,Saturday,52,3158,7493,123182.40,2368.89,116837.15
3,Wednesday,52,3024,6946,114408.40,2200.16,116837.15
4,Tuesday,52,2973,6895,114133.80,2194.88,116837.15
5,Monday,48,2794,6485,107329.55,2236.03,116837.15
6,Sunday,52,2624,6035,99203.50,1907.76,116837.15


### 8.8 Days With No Sales

`revenue_per_day` in the previous query is not cosmetic: the calendar is not complete, and the
gaps are not spread evenly across weekdays.

In [17]:
run("""
WITH RECURSIVE calendar(day) AS (
    SELECT '2015-01-01'
    UNION ALL
    SELECT date(day, '+1 day') FROM calendar WHERE day < '2015-12-31'
)
SELECT c.day AS missing_day,
       CASE strftime('%w', c.day)
           WHEN '0' THEN 'Sunday'   WHEN '1' THEN 'Monday'
           WHEN '2' THEN 'Tuesday'  WHEN '3' THEN 'Wednesday'
           WHEN '4' THEN 'Thursday' WHEN '5' THEN 'Friday'
           ELSE 'Saturday'
       END AS day_of_week
FROM calendar AS c
WHERE NOT EXISTS (SELECT 1 FROM orders AS o WHERE o.order_date = c.day)
ORDER BY c.day;
""")

,missing_day,day_of_week
0,2015-09-24,Thursday
1,2015-09-25,Friday
2,2015-10-05,Monday
3,2015-10-12,Monday
4,2015-10-19,Monday
5,2015-10-26,Monday
6,2015-12-25,Friday


### 8.9 Profitability by Month

In [18]:
run("""
SELECT strftime('%Y-%m', o.order_date)                        AS month,
       COUNT(DISTINCT o.order_date)                           AS operating_days,
       COUNT(DISTINCT o.order_id)                             AS total_orders,
       SUM(od.quantity)                                       AS total_quantity,
       ROUND(SUM(od.quantity * p.unit_price), 2)              AS total_revenue,
       ROUND(AVG(SUM(od.quantity * p.unit_price)) OVER (), 2) AS avg_revenue
FROM orders AS o
JOIN order_details AS od ON od.order_id = o.order_id
JOIN pizzas        AS p  ON p.pizza_id  = od.pizza_id
GROUP BY month
ORDER BY month;
""")

,month,operating_days,total_orders,total_quantity,total_revenue,avg_revenue
0,2015-01,31,1845,4232,69793.30,68155.0
1,2015-02,28,1685,3961,65159.60,68155.0
2,2015-03,31,1840,4261,70397.10,68155.0
3,2015-04,30,1799,4151,68736.80,68155.0
4,2015-05,31,1853,4328,71402.75,68155.0
5,2015-06,30,1773,4107,68230.20,68155.0
6,2015-07,31,1935,4392,72557.90,68155.0
7,2015-08,31,1841,4168,68278.25,68155.0
8,2015-09,28,1661,3890,64180.05,68155.0
9,2015-10,27,1646,3883,64027.60,68155.0


### 8.10 Performance by Pizza Category

In [19]:
run("""
SELECT p.pizza_category,
       SUM(od.quantity)                          AS total_quantity,
       ROUND(SUM(od.quantity * p.unit_price), 2) AS total_revenue,
       ROUND(100.0 * SUM(od.quantity * p.unit_price)
             / SUM(SUM(od.quantity * p.unit_price)) OVER (), 1) AS revenue_share_pct
FROM pizzas AS p
JOIN order_details AS od ON od.pizza_id = p.pizza_id
GROUP BY p.pizza_category
ORDER BY total_quantity DESC;
""")

,pizza_category,total_quantity,total_revenue,revenue_share_pct
0,Classic,14888,220053.10,26.9
1,Supreme,11987,208197.00,25.5
2,Veggie,11649,193690.45,23.7
3,Chicken,11050,195919.50,24.0


### 8.11 Performance by Pizza Size

In [20]:
run("""
SELECT p.pizza_size,
       COUNT(DISTINCT p.pizza_name)              AS pizzas_offered,
       SUM(od.quantity)                          AS total_quantity,
       ROUND(SUM(od.quantity * p.unit_price), 2) AS total_revenue,
       ROUND(100.0 * SUM(od.quantity * p.unit_price)
             / SUM(SUM(od.quantity * p.unit_price)) OVER (), 1) AS revenue_share_pct
FROM pizzas AS p
JOIN order_details AS od ON od.pizza_id = p.pizza_id
GROUP BY p.pizza_size
ORDER BY total_quantity DESC;
""")

,pizza_size,pizzas_offered,total_quantity,total_revenue,revenue_share_pct
0,L,30,18956,375318.70,45.9
1,M,29,15635,249382.25,30.5
2,S,30,14403,178076.50,21.8
3,XL,1,552,14076.00,1.7
4,XXL,1,28,1006.60,0.1


### 8.12 Top 5 Orders by Value

In [21]:
run("""
SELECT o.order_id,
       o.order_date,
       o.order_time,
       SUM(od.quantity)                          AS pizzas,
       ROUND(SUM(od.quantity * p.unit_price), 2) AS order_total
FROM orders AS o
JOIN order_details AS od ON od.order_id = o.order_id
JOIN pizzas        AS p  ON p.pizza_id  = od.pizza_id
GROUP BY o.order_id
ORDER BY order_total DESC
LIMIT 5;
""")

,order_id,order_date,order_time,pizzas,order_total
0,18845,2015-11-18,12:25:12,28,444.20
1,10760,2015-06-30,13:31:27,25,417.15
2,1096,2015-01-19,12:56:45,15,285.15
3,6169,2015-04-14,13:14:51,15,284.00
4,740,2015-01-13,12:29:51,15,280.95


### 8.13 Menu Structure: Sizes Offered and Entry Price

Sorted slowest-selling first. `entry_price` is the cheapest size a customer can buy the pizza in
- the price of entry to that item.

In [22]:
run("""
WITH menu AS (
    SELECT p.pizza_name,
           COUNT(DISTINCT p.pizza_size)        AS sizes_offered,
           GROUP_CONCAT(DISTINCT p.pizza_size) AS sizes,
           MIN(p.unit_price)                   AS entry_price,
           SUM(od.quantity)                    AS units_sold
    FROM pizzas AS p
    JOIN order_details AS od ON od.pizza_id = p.pizza_id
    GROUP BY p.pizza_name
)
SELECT pizza_name,
       sizes_offered,
       sizes,
       entry_price,
       ROUND(AVG(entry_price) OVER (), 2) AS menu_avg_entry_price,
       units_sold,
       RANK() OVER (ORDER BY units_sold)  AS slowest_rank
FROM menu
ORDER BY units_sold;
""")

,pizza_name,sizes_offered,sizes,entry_price,menu_avg_entry_price,units_sold,slowest_rank
0,The Brie Carre Pizza,1,S,23.65,12.79,490,1
1,The Mediterranean Pizza,3,"M,L,S",12.00,12.79,934,2
2,The Calabrese Pizza,3,"M,S,L",12.25,12.79,937,3
3,The Spinach Supreme Pizza,3,"S,M,L",12.50,12.79,950,4
4,The Soppressata Pizza,3,"L,M,S",12.50,12.79,961,5
5,The Spinach Pesto Pizza,3,"L,S,M",12.50,12.79,970,6
6,The Chicken Pesto Pizza,3,"L,M,S",12.75,12.79,973,7
7,The Italian Vegetables Pizza,3,"S,L,M",12.75,12.79,981,8
8,The Chicken Alfredo Pizza,3,"S,M,L",12.75,12.79,987,9
9,The Green Garden Pizza,3,"S,L,M",12.00,12.79,997,10


### 8.14 Line Items of the Largest Order

In [23]:
run("""
WITH biggest_order AS (
    SELECT od.order_id
    FROM order_details AS od
    JOIN pizzas AS p ON p.pizza_id = od.pizza_id
    GROUP BY od.order_id
    ORDER BY SUM(od.quantity * p.unit_price) DESC
    LIMIT 1
)
SELECT p.pizza_name,
       p.pizza_size,
       od.quantity,
       p.unit_price,
       ROUND(od.quantity * p.unit_price, 2) AS line_total,
       ROUND(SUM(od.quantity * p.unit_price) OVER (
           ORDER BY od.quantity * p.unit_price DESC, p.pizza_id
       ), 2)                                AS running_total,
       RANK() OVER (ORDER BY od.quantity * p.unit_price DESC) AS line_rank
FROM order_details AS od
JOIN pizzas AS p ON p.pizza_id = od.pizza_id
WHERE od.order_id = (SELECT order_id FROM biggest_order)
ORDER BY line_rank;
""")

,pizza_name,pizza_size,quantity,unit_price,line_total,running_total,line_rank
0,The Barbecue Chicken Pizza,M,3,16.75,50.25,50.25,1
1,The Thai Chicken Pizza,L,2,20.75,41.50,91.75,2
2,The Sicilian Pizza,M,2,16.25,32.50,124.25,3
3,The Greek Pizza,XL,1,25.50,25.50,149.75,4
4,The Pepperoni Pizza,M,2,12.50,25.00,174.75,5
5,The Big Meat Pizza,S,2,12.00,24.00,198.75,6
6,The Hawaiian Pizza,S,2,10.50,21.00,219.75,7
7,The Prosciutto and Arugula Pizza,L,1,20.75,20.75,240.50,8
8,The Southwest Chicken Pizza,L,1,20.75,20.75,261.25,8
9,The Vegetables + Vegetables Pizza,L,1,20.25,20.25,281.50,10


## 9. Insights

Total for 2015: **21,350 orders**, **49,574 pizzas**, **\$817,860** in revenue over **358 trading
days** - an average of **\$2,285 per day**. A typical order is **2.32 pizzas** worth **\$38.31**.

### Popular and Unpopular Pizzas

**Best sellers (units):** The Classic Deluxe (2,453), The Barbecue Chicken (2,432),
The Hawaiian (2,422), The Pepperoni (2,418), The Thai Chicken (2,371).

**Worst sellers (units):** The Brie Carre (490), The Mediterranean (934), The Calabrese (937),
The Spinach Supreme (950), The Soppressata (961).

The Brie Carre stands out: it sells roughly half as much as the next-worst pizza. A narrow
listing is not the explanation - it is one of three items sold in a single size, and the other two
(The Big Meat, S only; The Five Cheese, L only) move 1,914 and 1,409 units. Price is the more likely cause: the
cheapest size it can be bought in costs \$23.65, against a menu-wide average entry price of
\$12.79 and a next-highest of \$18.50.

### Sales by Hour

Trade is concentrated in two peaks - lunch (12:00-13:00) and dinner (17:00-19:00).

- **12:00-13:00** brings **\$217,944**, or **26.6%** of annual revenue, out of two hours a day.
- **17:00-19:00** adds **\$248,163** (30.3%).
- **09:00, 10:00 and 23:00 combined** produce **\$1,508 for the entire year** - **0.18%** of
  revenue, roughly 108x below the average hour.

09:00 is not a slow hour so much as a rounding error: across the whole year it contains **exactly
one order** - #19176, placed at 09:52 on 2015-11-24, four pizzas for \$83.

### Sales by Day of the Week

| Day | Revenue | Trading days | Revenue per trading day |
|---|---:|---:|---:|
| Friday | \$136,074 | 50 | \$2,721 |
| Thursday | \$123,529 | 52 | \$2,376 |
| Saturday | \$123,182 | 52 | \$2,369 |
| Monday | \$107,330 | 48 | \$2,236 |
| Wednesday | \$114,408 | 52 | \$2,200 |
| Tuesday | \$114,134 | 52 | \$2,195 |
| Sunday | \$99,204 | 52 | \$1,908 |

Friday is the clear best day, **16.5%** above the weekly average.

Monday ranks second-worst on total revenue, but it traded on four fewer days than the rest of
the week: the dataset is missing **all four Mondays in October**, so Monday is measured over 48
trading days instead of 52. Per trading day it sits mid-table, in fourth. **Sunday is the only
genuinely weak day** - **15.1%** below average on totals, and last on revenue per trading day as
well.

### Sales by Month

Revenue is flat across the year: every month lands between **\$64,028** and **\$72,558** against
an average of **\$68,155**, a spread of about +/-6%. July (\$72,558) and May (\$71,403) are the
strongest; October (\$64,028), September (\$64,180) and December (\$64,701) the weakest.

October and September are partly an artefact of the seven missing days (four in October, two in
September). December is genuine, and 25 December is the one day in the dataset that looks like a
deliberate closure.

**There is no usable seasonal signal here.** A +/-6% band across twelve months of a single year is
not something to plan against.

### Category and Size

| Category | Units | Revenue | Share |
|---|---:|---:|---:|
| Classic | 14,888 | \$220,053 | 26.9% |
| Supreme | 11,987 | \$208,197 | 25.5% |
| Chicken | 11,050 | \$195,920 | 24.0% |
| Veggie | 11,649 | \$193,690 | 23.7% |

The four categories are effectively tied. Classic leads on units by a wider margin than on
revenue because it is the cheapest category per pizza.

| Size | Units | Revenue | Share |
|---|---:|---:|---:|
| L | 18,956 | \$375,319 | 45.9% |
| M | 15,635 | \$249,382 | 30.5% |
| S | 14,403 | \$178,077 | 21.8% |
| XL | 552 | \$14,076 | 1.7% |
| XXL | 28 | \$1,007 | 0.1% |

Size is where the real concentration is: **L alone is 46% of revenue**, and L, M and S together
are **98.2%**. XL is offered on one pizza only (The Greek), XXL likewise, and XXL sold **28 units
in a year** - about one every two weeks.

## 10. Recommendations

### 1. Size operations around the lunch and dinner peaks
12:00-13:00 and 17:00-19:00 are 57% of revenue in five hours a day. Staffing, prep and delivery
capacity should be planned around those windows first; everything else is a secondary constraint.
Combo and pre-order offers work best aimed just *before* each peak (11:00 and 16:00), to flatten
the load rather than add to it.

### 2. Trim the opening hours at both ends
09:00, 10:00 and 23:00 generate \$1,508 a year between them - 0.18% of revenue, below the cost of
having anyone in the building. Opening at 11:00 and closing at 22:00 would give up that 0.18% and
remove roughly three staffed hours a day.

The caveat: this dataset has no cost, staffing or wage data, so the argument rests on revenue
alone. It is strong enough to justify a trial - close those hours for a quarter and check whether
the demand shifts to 11:00 or simply disappears.

### 3. Target Sunday, the weakest trading day
Sunday runs 15.1% below the weekly average on totals and is last on revenue per trading day as
well. It is the clearest candidate for a family or bundle offer.

Monday is a different case and should not be treated the same way. It looks weak on totals only
because four of its shifts are missing from the data; per trading day it is mid-table. Confirm
whether those four October Mondays were genuine closures or a recording gap before deciding
whether Monday needs anything at all.

### 4. Drop XXL, and review XL
XXL sold 28 units all year (\$1,007) and exists on a single pizza. It costs menu space, prep
training and inventory for 0.1% of revenue. XL is only marginally better at 1.7%.

The alternative reading is that both sizes are offered on one pizza only and so never had a fair
test. If the intent is to keep large formats, extend XL to the top five sellers and measure
again; if not, remove both and simplify the menu.

### 5. Re-price The Brie Carre, or drop it
At 490 units it is the slowest item on the menu, and the obvious excuse does not hold: it is not
handicapped by being sold in one size, because The Big Meat is also S-only and moves 1,914
units. What is unusual is the price. Its only size is a small at \$23.65, while the average entry
price on this menu is \$12.79 and the next dearest is \$18.50 - it is priced above every large on
the menu and sold as a small.

It also carries five ingredients that appear on no other pizza (brie carre cheese, prosciutto,
caramelised onions, pears, thyme), so it holds dedicated inventory for the slowest-moving dish.
Either test it at a price closer to its peers, or remove it and free that inventory.

### 6. Do not build a seasonal calendar yet
Monthly revenue varies by +/-6% with no discernible pattern, and part of that spread is missing
days rather than demand. A seasonal campaign calendar needs at least two more years of data before
it can be justified from this dataset.

### What this dataset cannot answer
There are no customer identifiers (so no repeat rate, basket affinity or cohort analysis), no
costs or margins (so "most profitable" here means *highest revenue*, not highest profit), no
staffing levels or delivery times, and no promotion history. The staffing and menu
recommendations above are directional until cost data is available.

---

In [24]:
con.close()
print("done")

done
